<a href="https://colab.research.google.com/github/derraan/Sonitude/blob/main/dsenet-colab-guide/python/dsenet/notebooks/dsenet_colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSENet Colab Training Workspace (Sonitude 6-mic)

This notebook trains an experimental DSENet variant for Sonitude's 6-microphone path, then exports a float32 streaming TensorFlow Lite model and metadata sidecar.

Scope and constraints:
- Uses a combined supervised synthetic dataset:
  - Dataset A: Paper-style LibriSpeech synthetic data (directional labels)
  - Dataset B: Sound Bubble metadata-prior synthetic data (directional labels)
- Sound Bubble packaged mixtures are used as acoustic priors only; they do not contain isolated references needed for DSENet supervised targets.
- This is a Sonitude adaptation (6 mics, 44.1 kHz), not a paper-faithful 3-mic 16 kHz reproduction.
- Training duration is user-controlled; no forced long training job.

Primary outputs:
- `dsenet_filter_estimator.keras`
- `dsenet_streaming_step.tflite`
- `dsenet_streaming_step.dsenet.json`
- Combined dataset manifests and logs

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Colab setup
# If this repo is not already present in /content, set REPO_URL and RUN_CLONE=True.
RUN_CLONE = True
REPO_URL = "https://github.com/derraan/Sonitude.git"
REPO_BRANCH = "docs/dsenet-colab-guide"
REPO_DIR = "/content/Sonitude"
if RUN_CLONE:
    !git clone -b "$REPO_BRANCH" --single-branch "$REPO_URL" "$REPO_DIR"

import os
import sys
import json
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

if not Path(REPO_DIR).exists():
    raise FileNotFoundError(f"Repo not found at {REPO_DIR}. Set RUN_CLONE=True or upload the repo.")
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

!python -m pip install -U pip wheel setuptools
!python -m pip install -r python/dsenet/requirements.txt
!python -m pip install torchaudio

import numpy as np
import soundfile as sf
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

fatal: destination path '/content/Sonitude' already exists and is not an empty directory.
Working directory: /content/Sonitude
TensorFlow: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# Notebook configuration
@dataclass
class ColabConfig:
    # Runtime/model
    sample_rate_hz: int = 44100
    num_mics: int = 6
    hop_size: int = 88      # ~2 ms at 44.1 kHz
    lookback: int = 88
    lookahead: int = 88
    hidden_size: int = 128

    # Training
    batch_size: int = 1
    epochs: int = 40
    learning_rate: float = 1e-3
    train_shuffle_buffer: int = 64

    # Data sizes (adjust up for real training)
    synth_a_train: int = 128
    synth_a_val: int = 16
    synth_a_test: int = 16
    synth_b_train: int = 128
    synth_b_val: int = 16
    synth_b_test: int = 16

    # Toggles
    use_synth_paper: bool = True
    use_soundbubble_style: bool = True

    # Paths
    work_root: str = "/content/dsenet_colab"
    librispeech_root: str = "/content/dsenet_colab/librispeech_raw"
    dataset_a_root: str = "/content/dsenet_colab/datasets/synth_paper_6mic"
    dataset_b_root: str = "/content/dsenet_colab/datasets/soundbubble_style_6mic"
    combined_root: str = "/content/dsenet_colab/datasets/combined"
    artifact_root: str = "/content/dsenet_colab/artifacts"

    # Optional Drive export path
    drive_root: str = "/content/drive/MyDrive/dsenet_colab_outputs"

CFG = ColabConfig()
for p in [CFG.work_root, CFG.librispeech_root, CFG.dataset_a_root, CFG.dataset_b_root, CFG.combined_root, CFG.artifact_root]:
    Path(p).mkdir(parents=True, exist_ok=True)

# Make local package importable
sys.path.insert(0, str(Path(REPO_DIR) / "python" / "dsenet"))

from dsenet.model import DSENetConfig, build_streaming_step_model
from dsenet.dataset import paper_target_from_reference_signals
from dsenet.losses import neg_si_sdr_loss
from dsenet.scale_recovery import estimate_eta

MODEL_CFG = DSENetConfig(
    sample_rate_hz=CFG.sample_rate_hz,
    num_mics=CFG.num_mics,
    hop_size=CFG.hop_size,
    lookback=CFG.lookback,
    lookahead=CFG.lookahead,
    hidden_size=CFG.hidden_size,
)
print(MODEL_CFG)

DSENetConfig(sample_rate_hz=44100, num_mics=6, hop_size=88, lookback=88, lookahead=88, hidden_size=128, l2_eps=1e-08, layer_norm_eps=1e-06)


In [5]:
# Sonitude 6-mic geometry from config/geometry_soundbubble_initial.yaml
GEOMETRY_IDS = [
    "M0_upper_inner_left",
    "M1_upper_inner_right",
    "M2_upper_outer_left",
    "M3_upper_outer_right",
    "M4_left_earcup",
    "M5_right_earcup",
]
GEOMETRY_XYZ_M = np.array([
    [-0.0380, 0.1680, 0.0000],
    [ 0.0380, 0.1680, 0.0000],
    [-0.1040, 0.1150, 0.0000],
    [ 0.1040, 0.1150, 0.0000],
    [-0.1295, 0.0000, 0.0000],
    [ 0.1295, 0.0000, 0.0000],
], dtype=np.float64)

assert GEOMETRY_XYZ_M.shape == (CFG.num_mics, 3)

def linear_resample(x: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    if sr_in == sr_out:
        return x.astype(np.float32)
    t_in = np.arange(x.shape[0], dtype=np.float64) / float(sr_in)
    n_out = int(round(x.shape[0] * (float(sr_out) / float(sr_in))))
    t_out = np.arange(n_out, dtype=np.float64) / float(sr_out)
    return np.interp(t_out, t_in, x.astype(np.float64)).astype(np.float32)

print("Geometry loaded for", CFG.num_mics, "mics")

Geometry loaded for 6 mics


In [6]:
# Download a small LibriSpeech subset for synthetic generation.
# Increase selections for real training runs.
import torchaudio

def ensure_librispeech_subset(root: Path):
    root.mkdir(parents=True, exist_ok=True)
    # train-clean-100 is commonly available and enough for synthetic pairing.
    _ = torchaudio.datasets.LIBRISPEECH(root=str(root), url="train-clean-100", download=True)
    _ = torchaudio.datasets.LIBRISPEECH(root=str(root), url="dev-clean", download=True)
    files = sorted(root.rglob("*.flac"))
    if len(files) < 50:
        raise RuntimeError(f"Expected at least 50 flac files, found {len(files)}")
    return files

librispeech_files = ensure_librispeech_subset(Path(CFG.librispeech_root))
print("LibriSpeech files:", len(librispeech_files))

LibriSpeech files: 31242


In [7]:
# Generalized synthetic dataset generator (6-mic capable, configurable priors)
try:
    import pyroomacoustics as pra
except Exception as exc:
    raise RuntimeError("pyroomacoustics is required; install from requirements.txt") from exc

@dataclass
class GenerationPriors:
    sample_rate_hz: int
    duration_s: float = 2.0
    room_l_min_m: float = 5.0
    room_l_max_m: float = 10.0
    room_h_min_m: float = 2.0
    room_h_max_m: float = 4.0
    rt60_min_s: float = 0.1
    rt60_max_s: float = 0.5
    target_az_min_deg: float = -10.0
    target_az_max_deg: float = 10.0
    masker_az_min_deg: float = -180.0
    masker_az_max_deg: float = 180.0
    source_r_min_m: float = 0.5
    source_r_max_m: float = 4.0
    sir_min_db: float = -5.0
    sir_max_db: float = 5.0
    sigma: float = 0.2
    rho: float = 8.0
    num_sources: int = 2


def load_mono_resampled(path: Path, sample_rate_hz: int, target_len: int) -> np.ndarray:
    x, sr = sf.read(str(path), dtype="float32")
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    x = linear_resample(x, sr_in=sr, sr_out=sample_rate_hz)
    if x.shape[0] < target_len:
        reps = int(np.ceil(target_len / x.shape[0]))
        x = np.tile(x, reps)
    return x[:target_len].astype(np.float32)


def polar_to_xyz(r_m: float, az_deg: float, origin_xyz: np.ndarray) -> np.ndarray:
    az = np.deg2rad(az_deg)
    return origin_xyz + np.array([r_m * np.sin(az), r_m * np.cos(az), 0.0], dtype=np.float64)


def sample_room_and_array(rng: np.random.Generator, priors: GenerationPriors, mic_positions_xyz_m: np.ndarray):
    for _ in range(20):
        room_l = rng.uniform(priors.room_l_min_m, priors.room_l_max_m)
        room_w = rng.uniform(priors.room_l_min_m, priors.room_l_max_m)
        room_h = rng.uniform(priors.room_h_min_m, priors.room_h_max_m)
        rt60 = rng.uniform(priors.rt60_min_s, priors.rt60_max_s)
        try:
            e_abs, max_order = pra.inverse_sabine(rt60, [room_l, room_w, room_h])
            center = np.array([room_l / 2.0, room_w / 2.0, 1.5], dtype=np.float64)
            mic_xyz_world = mic_positions_xyz_m + center[None, :]
            return room_l, room_w, room_h, rt60, e_abs, max_order, center, mic_xyz_world
        except ValueError:
            continue
    raise RuntimeError("Could not sample valid room parameters for inverse_sabine")


def simulate_multisource_example(
    priors: GenerationPriors,
    mic_positions_xyz_m: np.ndarray,
    speech_signals: List[np.ndarray],
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    (
        room_l,
        room_w,
        room_h,
        rt60,
        e_abs,
        max_order,
        array_center,
        mic_xyz_world,
    ) = sample_room_and_array(rng, priors, mic_positions_xyz_m)

    azimuths_deg = []
    radii_m = []
    src_positions = []
    for si in range(priors.num_sources):
        if si == 0:
            az = float(rng.uniform(priors.target_az_min_deg, priors.target_az_max_deg))
        else:
            az = float(rng.uniform(priors.masker_az_min_deg, priors.masker_az_max_deg))
        r = float(rng.uniform(priors.source_r_min_m, priors.source_r_max_m))
        azimuths_deg.append(az)
        radii_m.append(r)
        src_positions.append(polar_to_xyz(r, az, array_center))

    room_mix = pra.ShoeBox(
        [room_l, room_w, room_h],
        fs=priors.sample_rate_hz,
        materials=pra.Material(e_abs),
        max_order=max_order,
    )
    room_mix.add_microphone_array(mic_xyz_world.T)

    isolated_ref = []
    for si, sig in enumerate(speech_signals):
        room_mix.add_source(src_positions[si], signal=sig.astype(np.float32))

        room_si = pra.ShoeBox(
            [room_l, room_w, room_h],
            fs=priors.sample_rate_hz,
            materials=pra.Material(e_abs),
            max_order=max_order,
        )
        room_si.add_microphone_array(mic_xyz_world.T)
        room_si.add_source(src_positions[si], signal=sig.astype(np.float32))
        room_si.simulate()
        isolated_ref.append(room_si.mic_array.signals[0].astype(np.float32))

    room_mix.simulate()
    mix = room_mix.mic_array.signals.T.astype(np.float32)

    # Simple SIR scaling on maskers relative to target, applied at reference-mic component level.
    target_ref = isolated_ref[0]
    if len(isolated_ref) > 1:
        desired_sir_db = float(rng.uniform(priors.sir_min_db, priors.sir_max_db))
        p_t = float(np.mean(target_ref * target_ref) + 1e-8)
        for si in range(1, len(isolated_ref)):
            p_m = float(np.mean(isolated_ref[si] * isolated_ref[si]) + 1e-8)
            gain = np.sqrt(p_t / (p_m * (10.0 ** (desired_sir_db / 10.0))))
            isolated_ref[si] *= float(gain)

    min_len = min([mix.shape[0]] + [x.shape[0] for x in isolated_ref])
    mix = mix[:min_len]
    isolated_ref = [x[:min_len] for x in isolated_ref]

    azimuths_rad = np.deg2rad(np.array(azimuths_deg, dtype=np.float32))
    target = paper_target_from_reference_signals(
        np.stack(isolated_ref, axis=0), azimuths_rad, sigma=priors.sigma, rho=priors.rho
    ).astype(np.float32)

    meta = {
        "room_l": room_l,
        "room_w": room_w,
        "room_h": room_h,
        "rt60": rt60,
        "azimuths_deg": azimuths_deg,
        "radii_m": radii_m,
        "num_sources": priors.num_sources,
    }
    return mix, target, meta


def generate_split(
    files: List[Path],
    out_dir: Path,
    split_name: str,
    num_samples: int,
    priors: GenerationPriors,
    mic_positions_xyz_m: np.ndarray,
    seed: int,
):
    out_dir.mkdir(parents=True, exist_ok=True)
    n = int(round(priors.sample_rate_hz * priors.duration_s))
    rng = np.random.default_rng(seed)

    manifest = {
        "split": split_name,
        "config": asdict(priors),
        "num_mics": int(mic_positions_xyz_m.shape[0]),
        "samples": [],
    }

    for i in range(num_samples):
        pick = rng.choice(len(files), size=priors.num_sources, replace=False)
        src_paths = [files[int(ix)] for ix in pick]
        signals = [load_mono_resampled(p, priors.sample_rate_hz, n) for p in src_paths]
        mix, target, meta = simulate_multisource_example(priors, mic_positions_xyz_m, signals, rng)

        mix_path = out_dir / f"mix_{i:05d}.npy"
        tgt_path = out_dir / f"target_{i:05d}.npy"
        np.save(mix_path, mix)
        np.save(tgt_path, target)

        manifest["samples"].append(
            {
                "id": f"{split_name}_{i:05d}",
                "mixture_npy": mix_path.name,
                "target_npy": tgt_path.name,
                "src_files": [str(p) for p in src_paths],
                "meta": meta,
            }
        )

    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Generated {split_name}: {num_samples} samples -> {out_dir}")

In [ ]:
# Dataset A: paper-style priors, Sonitude 6-mic geometry, 44.1 kHz
paper_priors = GenerationPriors(
    sample_rate_hz=CFG.sample_rate_hz,
    duration_s=4.0,
    room_l_min_m=5.0,
    room_l_max_m=10.0,
    room_h_min_m=2.0,
    room_h_max_m=4.0,
    rt60_min_s=0.1,
    rt60_max_s=0.5,
    target_az_min_deg=-10.0,
    target_az_max_deg=10.0,
    masker_az_min_deg=-180.0,
    masker_az_max_deg=180.0,
    source_r_min_m=0.5,
    source_r_max_m=2.0,
    sir_min_db=-5.0,
    sir_max_db=5.0,
    sigma=0.2,
    rho=8.0,
    num_sources=2,
)

if CFG.use_synth_paper:
    a_root = Path(CFG.dataset_a_root)
    generate_split(librispeech_files, a_root / "train", "train", CFG.synth_a_train, paper_priors, GEOMETRY_XYZ_M, seed=101)
    generate_split(librispeech_files, a_root / "val", "val", CFG.synth_a_val, paper_priors, GEOMETRY_XYZ_M, seed=102)
    generate_split(librispeech_files, a_root / "test", "test", CFG.synth_a_test, paper_priors, GEOMETRY_XYZ_M, seed=103)

In [ ]:
# Mount Google Drive and extract Sound Bubble metadata priors
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# Update if your Sound Bubble dataset is in a different Drive location.
SOUNDBUBBLE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_bridge/datasets/syn_1_5m/syn_1_5m')
assert SOUNDBUBBLE_ROOT.exists(), f"Missing path: {SOUNDBUBBLE_ROOT}"

def extract_soundbubble_priors(sb_root: Path, sample_limit: int = 400) -> Dict:
    meta_files = []
    for p in sb_root.rglob('metadata.json'):
        meta_files.append(p)
        if len(meta_files) >= sample_limit:
            break
    if not meta_files:
        raise RuntimeError(f"No metadata.json found under {sb_root}")

    rt60_vals = []
    room_l_vals = []
    room_w_vals = []
    room_h_vals = []
    target_angles = []
    masker_angles = []
    radii = []
    n_out_vals = []

    for mf in meta_files:
        d = json.loads(mf.read_text(encoding='utf-8'))
        room = d.get('room_info', {})
        walls = room.get('walls', None)
        if isinstance(walls, list) and len(walls) == 4:
            # walls format seen as [xmin, xmax, ymax, ymin]
            room_l_vals.append(float(abs(walls[1] - walls[0])))
            room_w_vals.append(float(abs(walls[2] - walls[3])))
        if 'rt60' in room:
            rt60_vals.append(float(room['rt60']))

        # Estimate room height from mic/source z extents when explicit height not present
        z_vals = []
        for k, v in d.items():
            if isinstance(v, dict) and 'position' in v and len(v['position']) == 3:
                z_vals.append(float(v['position'][2]))
        if z_vals:
            room_h_vals.append(max(2.0, min(4.0, float(np.max(z_vals) + 2.0))))

        voices = sorted([k for k in d.keys() if k.startswith('voice')])
        mic0 = d.get('mic00', {}).get('position', None)
        if mic0 is not None and len(voices) >= 1:
            m = np.array(mic0, dtype=np.float64)
            for i, vk in enumerate(voices):
                pos = np.array(d[vk]['position'], dtype=np.float64)
                rel = pos - m
                az = math.degrees(math.atan2(rel[0], rel[1]))
                rr = float(np.linalg.norm(rel))
                radii.append(rr)
                if i == 0:
                    target_angles.append(az)
                else:
                    masker_angles.append(az)

        n_out_vals.append(int(d.get('n_out', max(1, len(voices)))))

    if not rt60_vals:
        rt60_vals = [0.1, 0.5]
    if not room_l_vals:
        room_l_vals = [5.0, 10.0]
    if not room_w_vals:
        room_w_vals = [5.0, 10.0]
    if not room_h_vals:
        room_h_vals = [2.0, 4.0]
    if not target_angles:
        target_angles = [-10.0, 10.0]
    if not masker_angles:
        masker_angles = [-180.0, 180.0]
    if not radii:
        radii = [0.5, 4.0]

    pri = {
        'room_l_min_m': float(np.percentile(room_l_vals, 5)),
        'room_l_max_m': float(np.percentile(room_l_vals, 95)),
        'room_h_min_m': float(np.percentile(room_h_vals, 5)),
        'room_h_max_m': float(np.percentile(room_h_vals, 95)),
        'rt60_min_s': float(np.percentile(rt60_vals, 5)),
        'rt60_max_s': float(np.percentile(rt60_vals, 95)),
        'target_az_min_deg': float(np.percentile(target_angles, 5)),
        'target_az_max_deg': float(np.percentile(target_angles, 95)),
        'masker_az_min_deg': float(np.percentile(masker_angles, 5)),
        'masker_az_max_deg': float(np.percentile(masker_angles, 95)),
        'source_r_min_m': float(np.percentile(radii, 5)),
        'source_r_max_m': float(np.percentile(radii, 95)),
        'num_sources': int(max(2, int(round(np.median(n_out_vals))) + 1)),
    }
    return pri

sb_priors_stats = extract_soundbubble_priors(SOUNDBUBBLE_ROOT, sample_limit=400)
print(json.dumps(sb_priors_stats, indent=2))

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/colab_bridge/datasets/syn_1_5m/syn_1_5m" | head

In [ ]:
# Dataset B: Sound-Bubble-style priors, generated with valid DSENet targets.
# To mimic SB closer, generate at 24 kHz and then resample outputs to 44.1 kHz.

def resample_split_to_sr(split_dir: Path, sr_in: int, sr_out: int):
    manifest_path = split_dir / 'manifest.json'
    m = json.loads(manifest_path.read_text(encoding='utf-8'))
    for s in m['samples']:
        mix_p = split_dir / s['mixture_npy']
        tgt_p = split_dir / s['target_npy']
        mix = np.load(mix_p)
        tgt = np.load(tgt_p)
        mix_rs = np.stack([linear_resample(mix[:, ch], sr_in, sr_out) for ch in range(mix.shape[1])], axis=1)
        tgt_rs = linear_resample(tgt, sr_in, sr_out)
        n = min(mix_rs.shape[0], tgt_rs.shape[0])
        np.save(mix_p, mix_rs[:n].astype(np.float32))
        np.save(tgt_p, tgt_rs[:n].astype(np.float32))
    m['config']['sample_rate_hz_original'] = sr_in
    m['config']['sample_rate_hz_resampled'] = sr_out
    m['config']['resample_note'] = 'linear interpolation'
    manifest_path.write_text(json.dumps(m, indent=2), encoding='utf-8')

if CFG.use_soundbubble_style:
    sb24 = GenerationPriors(
        sample_rate_hz=24000,
        duration_s=4.0,
        room_l_min_m=sb_priors_stats['room_l_min_m'],
        room_l_max_m=sb_priors_stats['room_l_max_m'],
        room_h_min_m=sb_priors_stats['room_h_min_m'],
        room_h_max_m=sb_priors_stats['room_h_max_m'],
        rt60_min_s=sb_priors_stats['rt60_min_s'],
        rt60_max_s=sb_priors_stats['rt60_max_s'],
        target_az_min_deg=sb_priors_stats['target_az_min_deg'],
        target_az_max_deg=sb_priors_stats['target_az_max_deg'],
        masker_az_min_deg=sb_priors_stats['masker_az_min_deg'],
        masker_az_max_deg=sb_priors_stats['masker_az_max_deg'],
        #source_r_min_m=sb_priors_stats['source_r_min_m'],
        #source_r_max_m=sb_priors_stats['source_r_max_m'],
        source_r_min_m=0.5,
        source_r_max_m=2.0,
        sir_min_db=-5.0,
        sir_max_db=5.0,
        sigma=0.2,
        rho=8.0,
        num_sources=2,
    )

    b_root = Path(CFG.dataset_b_root)
    generate_split(librispeech_files, b_root / 'train', 'train', CFG.synth_b_train, sb24, GEOMETRY_XYZ_M, seed=201)
    generate_split(librispeech_files, b_root / 'val', 'val', CFG.synth_b_val, sb24, GEOMETRY_XYZ_M, seed=202)
    generate_split(librispeech_files, b_root / 'test', 'test', CFG.synth_b_test, sb24, GEOMETRY_XYZ_M, seed=203)

    for split in ['train', 'val', 'test']:
        resample_split_to_sr(b_root / split, sr_in=24000, sr_out=CFG.sample_rate_hz)
    print('Dataset B generated and resampled to', CFG.sample_rate_hz)

In [ ]:
# Build combined manifests and tf.data loaders

def load_manifest(path: Path) -> Dict:
    return json.loads(path.read_text(encoding='utf-8'))


def collect_samples(dataset_roots: List[Path], split: str) -> List[Tuple[Path, Path, str]]:
    out = []
    for r in dataset_roots:
        mp = r / split / 'manifest.json'
        if not mp.exists():
            continue
        man = load_manifest(mp)
        for s in man['samples']:
            out.append((r / split / s['mixture_npy'], r / split / s['target_npy'], s['id']))
    if not out:
        raise RuntimeError(f'No samples found for split={split}')
    return out


def frame_sample_for_dsenet(mix_tm: np.ndarray, target_t: np.ndarray, cfg: DSENetConfig):
    # mix_tm: [T, M], target_t: [T]
    T, M = mix_tm.shape
    assert M == cfg.num_mics
    frame_len = cfg.hop_size + cfg.lookback + cfg.lookahead
    filter_len = 1 + cfg.lookback + cfg.lookahead
    needed = frame_len + filter_len - 1
    if T < needed:
        return None, None

    num_hops = 1 + (T - needed) // cfg.hop_size
    frame_seq = np.zeros((num_hops, cfg.num_mics, frame_len), dtype=np.float32)
    target_seq = np.zeros((num_hops, cfg.hop_size), dtype=np.float32)

    for k in range(num_hops):
        base = k * cfg.hop_size
        frame = mix_tm[base:base + frame_len, :].T
        frame_seq[k] = frame

        # Supervision aligned to the produced hop output interval.
        t0 = base + cfg.lookback
        t1 = t0 + cfg.hop_size
        if t1 > target_t.shape[0]:
            break
        target_seq[k] = target_t[t0:t1]

    return frame_seq, target_seq


def build_numpy_split(sample_tuples: List[Tuple[Path, Path, str]], cfg: DSENetConfig):
    frame_batches = []
    target_batches = []
    sample_ids = []

    for mix_path, tgt_path, sid in sample_tuples:
        mix = np.load(mix_path).astype(np.float32)
        tgt = np.load(tgt_path).astype(np.float32)
        n = min(mix.shape[0], tgt.shape[0])
        mix = mix[:n]
        tgt = tgt[:n]
        fs, ts = frame_sample_for_dsenet(mix, tgt, cfg)
        if fs is None:
            continue
        frame_batches.append(fs)
        target_batches.append(ts)
        sample_ids.append(sid)

    if not frame_batches:
        raise RuntimeError('No framed examples could be created')

    # Keep one training item per original utterance with shape [num_hops, ...].
    # pad to max hops for batching.
    max_hops = max(x.shape[0] for x in frame_batches)
    X = np.zeros((len(frame_batches), max_hops, cfg.num_mics, cfg.hop_size + cfg.lookback + cfg.lookahead), dtype=np.float32)
    Y = np.zeros((len(frame_batches), max_hops, cfg.hop_size), dtype=np.float32)
    mask = np.zeros((len(frame_batches), max_hops), dtype=np.float32)

    for i, (fs, ts) in enumerate(zip(frame_batches, target_batches)):
        h = fs.shape[0]
        X[i, :h] = fs
        Y[i, :h] = ts
        mask[i, :h] = 1.0

    return X, Y, mask, sample_ids


def make_tf_dataset(X: np.ndarray, Y: np.ndarray, mask: np.ndarray, batch_size: int, training: bool):
    ds = tf.data.Dataset.from_tensor_slices(({'frame_seq': X, 'mask_seq': mask}, Y))
    if training:
        ds = ds.shuffle(min(len(X), CFG.train_shuffle_buffer), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

roots = []
if CFG.use_synth_paper:
    roots.append(Path(CFG.dataset_a_root))
if CFG.use_soundbubble_style:
    roots.append(Path(CFG.dataset_b_root))

train_samples = collect_samples(roots, 'train')
val_samples = collect_samples(roots, 'val')
test_samples = collect_samples(roots, 'test')

X_train, Y_train, M_train, train_ids = build_numpy_split(train_samples, MODEL_CFG)
X_val, Y_val, M_val, val_ids = build_numpy_split(val_samples, MODEL_CFG)
X_test, Y_test, M_test, test_ids = build_numpy_split(test_samples, MODEL_CFG)

train_ds = make_tf_dataset(X_train, Y_train, M_train, CFG.batch_size, training=True)
val_ds = make_tf_dataset(X_val, Y_val, M_val, CFG.batch_size, training=False)

print('Train:', X_train.shape, Y_train.shape)
print('Val  :', X_val.shape, Y_val.shape)
print('Test :', X_test.shape, Y_test.shape)

In [ ]:
# Build in-graph training model: feature stack + GRU filter estimation + TF filter interpolation + FaS
class FilterAndSumInterpolationLayer(tf.keras.layers.Layer):
    def __init__(self, num_mics: int, hop_size: int, lookback: int, lookahead: int, **kwargs):
        super().__init__(**kwargs)
        self.num_mics = int(num_mics)
        self.hop_size = int(hop_size)
        self.lookback = int(lookback)
        self.lookahead = int(lookahead)
        self.filter_len = 1 + self.lookback + self.lookahead

    def call(self, inputs):
        frame_seq, h_seq_flat, mask_seq = inputs
        # frame_seq: [B,K,M,L+Lp+Lf]
        # h_seq_flat: [B,K,M*F]
        # mask_seq: [B,K]
        h_seq = tf.reshape(h_seq_flat, [tf.shape(h_seq_flat)[0], tf.shape(h_seq_flat)[1], self.num_mics, self.filter_len])

        # Build per-sample windows for each hop: [B,K,M,L,F]
        segments = tf.signal.frame(frame_seq, frame_length=self.filter_len, frame_step=1, axis=-1)

        B = tf.shape(frame_seq)[0]
        K = tf.shape(frame_seq)[1]
        L = self.hop_size
        t = tf.cast(tf.range(1, L + 1), tf.float32) / float(L)
        t = tf.reshape(t, [1, L, 1, 1])

        prev_h = tf.zeros([B, self.num_mics, self.filter_len], dtype=frame_seq.dtype)
        ta = tf.TensorArray(dtype=frame_seq.dtype, size=K)

        def cond(k, prev_h_state, out_ta):
            return k < K

        def body(k, prev_h_state, out_ta):
            cur_h = h_seq[:, k, :, :]            # [B,M,F]
            seg_k = segments[:, k, :, :, :]      # [B,M,L,F]
            seg_k = tf.transpose(seg_k, [0, 2, 1, 3])  # [B,L,M,F]

            h_interp = prev_h_state[:, None, :, :] + (cur_h - prev_h_state)[:, None, :, :] * t
            y_k = tf.reduce_sum(h_interp * seg_k, axis=[2, 3])  # [B,L]

            mk = tf.expand_dims(mask_seq[:, k], axis=-1)        # [B,1]
            y_k = y_k * tf.cast(mk, y_k.dtype)

            out_ta = out_ta.write(k, y_k)
            return k + 1, cur_h, out_ta

        _, _, ta = tf.while_loop(cond, body, [tf.constant(0), prev_h, ta], parallel_iterations=1)
        y = ta.stack()                    # [K,B,L]
        y = tf.transpose(y, [1, 0, 2])    # [B,K,L]
        return y


def build_training_model(cfg: DSENetConfig) -> tf.keras.Model:
    frame_len = cfg.hop_size + cfg.lookback + cfg.lookahead

    frame_seq_in = tf.keras.Input(shape=(None, cfg.num_mics, frame_len), name='frame_seq')
    mask_seq_in = tf.keras.Input(shape=(None,), name='mask_seq')

    # p_k is flattened frame-by-mic context as in the paper feature definition.
    p_seq = tf.keras.layers.Lambda(
        lambda t: tf.reshape(t, [tf.shape(t)[0], tf.shape(t)[1], cfg.feature_len]),
        output_shape=lambda s: (s[0], s[1], cfg.feature_len),
        name='pk_from_frame',
    )(frame_seq_in)

    x = tf.keras.layers.Lambda(
        lambda t: tf.reshape(t, [-1, cfg.feature_len]),
        output_shape=lambda s: (None, cfg.feature_len),
        name='bt_flat',
    )(p_seq)

    x = tf.keras.layers.Lambda(lambda t: t / (tf.sqrt(tf.reduce_sum(tf.square(t), axis=-1, keepdims=True) + cfg.l2_eps)), name='l2norm')(x)
    x = tf.keras.layers.Dense(cfg.hidden_size, name='feature_fc')(x)
    x = tf.keras.layers.PReLU(shared_axes=None, name='feature_prelu')(x)
    x = tf.keras.layers.LayerNormalization(epsilon=cfg.layer_norm_eps, name='feature_ln')(x)
    x = tf.keras.layers.Lambda(
        lambda args: tf.reshape(
            args[0], [tf.shape(args[1])[0], tf.shape(args[1])[1], cfg.hidden_size]
        ),
        output_shape=lambda s: (s[1][0], s[1][1], cfg.hidden_size),
        name='to_seq',
    )([x, p_seq])

    x = tf.keras.layers.GRU(cfg.hidden_size, return_sequences=True, reset_after=True, name='gru1')(x)
    x = tf.keras.layers.GRU(cfg.hidden_size, return_sequences=True, reset_after=True, name='gru2')(x)

    h_flat = tf.keras.layers.Lambda(
        lambda t: tf.reshape(t, [-1, cfg.hidden_size]),
        output_shape=lambda s: (None, cfg.hidden_size),
        name='gru_bt_flat',
    )(x)
    h_flat = tf.keras.layers.Dense(cfg.filters_flat_len, name='filter_fc')(h_flat)
    h_flat = tf.keras.layers.Lambda(
        lambda args: tf.reshape(
            args[0], [tf.shape(args[1])[0], tf.shape(args[1])[1], cfg.filters_flat_len]
        ),
        output_shape=lambda s: (s[1][0], s[1][1], cfg.filters_flat_len),
        name='hk_seq',
    )([h_flat, p_seq])

    y_hat = FilterAndSumInterpolationLayer(
        num_mics=cfg.num_mics,
        hop_size=cfg.hop_size,
        lookback=cfg.lookback,
        lookahead=cfg.lookahead,
        name='fas_interp',
    )([frame_seq_in, h_flat, mask_seq_in])

    return tf.keras.Model(inputs={'frame_seq': frame_seq_in, 'mask_seq': mask_seq_in}, outputs=y_hat, name='dsenet_training_model')

training_model = build_training_model(MODEL_CFG)
training_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=CFG.learning_rate), loss=neg_si_sdr_loss)
training_model.summary()

In [ ]:
# Train with LR decay every 2 epochs and checkpoint best val loss
ckpt_dir = Path(CFG.artifact_root) / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_path = ckpt_dir / 'best_training_model.weights.h5'

lr_schedule = tf.keras.callbacks.LearningRateScheduler(
    lambda epoch, lr: lr * (0.98 if (epoch > 0 and epoch % 2 == 0) else 1.0)
)
ckpt = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(best_path),
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True,
)

history = training_model.fit(
    {'frame_seq': X_train, 'mask_seq': M_train},
    Y_train,
    sample_weight=M_train,
    validation_data=({'frame_seq': X_val, 'mask_seq': M_val}, Y_val, M_val),
    epochs=CFG.epochs,
    batch_size=CFG.batch_size,
    callbacks=[lr_schedule, ckpt],
    verbose=1,
)

training_model.load_weights(best_path)
print('Loaded best weights:', best_path)
print('Loaded best model:', best_path)

In [ ]:
# Estimate eta on validation data (scale recovery)
val_pred = training_model.predict({'frame_seq': X_val, 'mask_seq': M_val}, batch_size=CFG.batch_size, verbose=0)
val_true_flat = []
val_pred_flat = []
for i in range(val_pred.shape[0]):
    valid_hops = int(np.sum(M_val[i]))
    if valid_hops <= 0:
        continue
    val_true_flat.append(Y_val[i, :valid_hops].reshape(-1))
    val_pred_flat.append(val_pred[i, :valid_hops].reshape(-1))

val_true_flat = np.concatenate(val_true_flat).astype(np.float32)
val_pred_flat = np.concatenate(val_pred_flat).astype(np.float32)
eta = float(estimate_eta(val_true_flat, val_pred_flat))
print('Estimated eta:', eta)

In [ ]:
# Transfer learned weights by layer name, save estimator, export streaming float32 TFLite + metadata
from dsenet.model import build_dsenet_filter_estimator
import hashlib

def copy_named_weights(src: tf.keras.Model, dst: tf.keras.Model, names: List[str]):
    for name in names:
        sw = src.get_layer(name).get_weights()
        if not sw:
            continue
        dst.get_layer(name).set_weights(sw)

shared_layer_names = [
    'l2norm',
    'feature_fc',
    'feature_prelu',
    'feature_ln',
    'gru1',
    'gru2',
    'filter_fc',
]

# 1) Save a trained one-step estimator model (for analysis/tracking)
estimator_model = build_dsenet_filter_estimator(MODEL_CFG)
_ = estimator_model(np.zeros((1, MODEL_CFG.feature_len), np.float32))
copy_named_weights(training_model, estimator_model, shared_layer_names)

artifact_root = Path(CFG.artifact_root)
artifact_root.mkdir(parents=True, exist_ok=True)
estimator_path = artifact_root / 'dsenet_filter_estimator.keras'
estimator_model.save(estimator_path)
print('Saved:', estimator_path)

# 2) Build streaming step model and copy same learned blocks
streaming_model = build_streaming_step_model(MODEL_CFG)
_ = streaming_model([
    np.zeros((1, MODEL_CFG.feature_len), np.float32),
    np.zeros((1, MODEL_CFG.hidden_size), np.float32),
    np.zeros((1, MODEL_CFG.hidden_size), np.float32),
])
copy_named_weights(training_model, streaming_model, shared_layer_names)

# 3) Convert to float32 TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(streaming_model)
converter.target_spec.supported_types = [tf.float32]
tflite_bytes = converter.convert()

tflite_path = artifact_root / 'dsenet_streaming_step.tflite'
tflite_path.write_bytes(tflite_bytes)

h = hashlib.sha256()
h.update(tflite_bytes)
sha = h.hexdigest()

metadata = {
    'schema_version': 1,
    'sample_rate_hz': MODEL_CFG.sample_rate_hz,
    'num_mics': MODEL_CFG.num_mics,
    'reference_mic_index': 0,
    'L': MODEL_CFG.hop_size,
    'Lp': MODEL_CFG.lookback,
    'Lf': MODEL_CFG.lookahead,
    'H': MODEL_CFG.hidden_size,
    'eta': eta,
    'trained': True,
    'tensor_names': {
        'p_k': 'serving_default_p_k:0',
        'gru1_state_in': 'serving_default_gru1_state_in:0',
        'gru2_state_in': 'serving_default_gru2_state_in:0',
        'h_k': 'StatefulPartitionedCall:0',
        'gru1_state_out': 'StatefulPartitionedCall:1',
        'gru2_state_out': 'StatefulPartitionedCall:2',
    },
    'tensor_shapes': {
        'p_k': [1, MODEL_CFG.feature_len],
        'gru1_state': [1, MODEL_CFG.hidden_size],
        'gru2_state': [1, MODEL_CFG.hidden_size],
        'h_k': [1, MODEL_CFG.filters_flat_len],
    },
    'model_sha256': sha,
    'model_filename': tflite_path.name,
    'geometry': {
        'ids': GEOMETRY_IDS,
        'xyz_m': GEOMETRY_XYZ_M.tolist(),
    },
    'config': {
        'sample_rate_hz': MODEL_CFG.sample_rate_hz,
        'num_mics': MODEL_CFG.num_mics,
        'hop_size': MODEL_CFG.hop_size,
        'lookback': MODEL_CFG.lookback,
        'lookahead': MODEL_CFG.lookahead,
        'hidden_size': MODEL_CFG.hidden_size,
        'l2_eps': MODEL_CFG.l2_eps,
        'layer_norm_eps': MODEL_CFG.layer_norm_eps,
    },
}

metadata_path = artifact_root / 'dsenet_streaming_step.dsenet.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved:', tflite_path)
print('Saved:', metadata_path)

In [ ]:
# Streaming parity gate (Keras streaming vs TFLite streaming), including mid-stream reset
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
in_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

rng = np.random.default_rng(1234)
s1 = np.zeros((1, MODEL_CFG.hidden_size), np.float32)
s2 = np.zeros((1, MODEL_CFG.hidden_size), np.float32)
max_abs = 0.0

num_hops = 1000
for hop in range(num_hops):
    pk = rng.standard_normal((1, MODEL_CFG.feature_len), dtype=np.float32)

    hk_k, s1_k, s2_k = streaming_model([pk, s1, s2], training=False)
    hk_ref = hk_k.numpy()
    s1_ref = s1_k.numpy()
    s2_ref = s2_k.numpy()

    interpreter.set_tensor(in_details[0]['index'], pk)
    interpreter.set_tensor(in_details[1]['index'], s1)
    interpreter.set_tensor(in_details[2]['index'], s2)
    interpreter.invoke()
    hk_tfl = interpreter.get_tensor(out_details[0]['index'])
    s1_tfl = interpreter.get_tensor(out_details[1]['index'])
    s2_tfl = interpreter.get_tensor(out_details[2]['index'])

    np.testing.assert_allclose(hk_tfl, hk_ref, rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(s1_tfl, s1_ref, rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(s2_tfl, s2_ref, rtol=1e-5, atol=1e-6)

    max_abs = max(max_abs, float(np.max(np.abs(hk_tfl - hk_ref))))

    s1 = s1_tfl
    s2 = s2_tfl
    if hop == num_hops // 2:
        s1.fill(0.0)
        s2.fill(0.0)

print(f'Parity passed over {num_hops} hops; max_abs={max_abs:.6e}')

In [ ]:
# Optional quick held-out metrics on test split
from dsenet.metrics import pair_metrics

test_pred = training_model.predict({'frame_seq': X_test, 'mask_seq': M_test}, batch_size=CFG.batch_size, verbose=0)

snr_vals = []
sisdr_vals = []
for i in range(test_pred.shape[0]):
    valid_hops = int(np.sum(M_test[i]))
    if valid_hops <= 0:
        continue
    y_true = Y_test[i, :valid_hops].reshape(-1)
    y_hat = (test_pred[i, :valid_hops].reshape(-1) * eta).astype(np.float32)
    m = pair_metrics(reference=y_true.astype(np.float32), estimate=y_hat)
    snr_vals.append(float(m.snr_db))
    sisdr_vals.append(float(m.si_sdr_db))

print('Test mean SNR  :', float(np.mean(snr_vals)) if snr_vals else None)
print('Test mean SI-SDR:', float(np.mean(sisdr_vals)) if sisdr_vals else None)

# Persist a compact combined manifest used in this run.
combined_manifest = {
    'config': asdict(CFG),
    'model_config': {
        'sample_rate_hz': MODEL_CFG.sample_rate_hz,
        'num_mics': MODEL_CFG.num_mics,
        'hop_size': MODEL_CFG.hop_size,
        'lookback': MODEL_CFG.lookback,
        'lookahead': MODEL_CFG.lookahead,
        'hidden_size': MODEL_CFG.hidden_size,
    },
    'geometry_ids': GEOMETRY_IDS,
    'geometry_xyz_m': GEOMETRY_XYZ_M.tolist(),
    'splits': {
        'train_ids': train_ids,
        'val_ids': val_ids,
        'test_ids': test_ids,
    },
}
combined_manifest_path = Path(CFG.combined_root) / 'combined_manifest.json'
combined_manifest_path.parent.mkdir(parents=True, exist_ok=True)
combined_manifest_path.write_text(json.dumps(combined_manifest, indent=2), encoding='utf-8')
print('Saved:', combined_manifest_path)

In [ ]:
# Save notebook artifacts to Google Drive (optional)
from shutil import copy2

drive_out = Path(CFG.drive_root)
drive_out.mkdir(parents=True, exist_ok=True)

for p in [
    estimator_path,
    tflite_path,
    metadata_path,
    combined_manifest_path,
]:
    copy2(p, drive_out / p.name)
    print('Copied:', p.name, '->', drive_out)

print('Artifact export complete.')

## Remaining Blockers and Caveats

- Sound Bubble packaged splits contain `mixture.wav` + `metadata.json` only; they do not contain isolated per-source reference signals needed for direct DSENet supervision.
- This notebook uses Sound Bubble metadata as an acoustic prior to synthesize valid supervised training targets from LibriSpeech.
- The Sound-Bubble-style subset is generated at 24 kHz and linearly resampled to 44.1 kHz to match Sonitude runtime validation.
- This is a Sonitude 6-mic adaptation, not a strict 3-mic Pixel-3 16 kHz paper reproduction.
- Pi 5 real-time guarantees are not established by Colab training; run Sonitude's native benchmark (`sonitude_dsenet_bench`) on Pi 5 for p50/p95/p99/max and deadline misses.
- If final deployment metadata validation fails in C++, confirm geometry IDs/order, sample rate, `L/Lp/Lf/H`, and tensor names/shapes exactly match runtime expectations.